[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Data_Quality_Tools.ipynb)

Playbook: [Data Quality Tools](../../playbooks/data/pipelines/data-quality/README.md)

# Data Quality Tools — Hands-On

**Automate data quality checks with Great Expectations + custom Python. Run quality gates that catch problems before downstream consumers do.**

---

## What You Will Do

1. Create sample call center data (with intentional quality issues)
2. Run quality checks with **custom Python** (no framework — understand the fundamentals)
3. Run quality checks with **Great Expectations** (production framework)
4. Compare: what each approach catches and how it reports
5. Integrate quality checks into a pipeline (quality gate pattern)

**Time:** ~20 minutes

**Prerequisites:** Basic Python. No cloud account needed.

---

## 1. Setup

In [ ]:
!pip install -q great_expectations pandas

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Setup complete")

---

## 2. Create Sample Data (With Intentional Problems)

This data has 5 intentional quality issues. Your quality checks should catch all 5.

In [ ]:
# WHY: Real data has quality issues. This simulates what your pipeline
# will encounter: nulls, duplicates, out-of-range values, invalid enums,
# and future timestamps.

calls = pd.DataFrame({
    "call_id": ["C-001", "C-002", "C-003", "C-004", "C-005",
                "C-006", "C-007", None, "C-009", "C-003"],  # Issue 1: null call_id (row 8)
                                                               # Issue 2: duplicate C-003 (row 10)
    "customer_id": ["CUST-100", "CUST-101", "CUST-102", "CUST-103", "CUST-104",
                    "CUST-105", "CUST-106", "CUST-107", "CUST-108", "CUST-102"],
    "status": ["resolved", "resolved", "missed", "resolved", "voicemail",
               "INVALID_STATUS", "resolved", "resolved", "in-progress", "missed"],  # Issue 3: invalid status (row 6)
    "duration": [340, 120, 0, 480, 60,
                 200, -50, 300, 0, 0],  # Issue 4: negative duration (row 7)
    "campaign_id": ["CAMP-A", "CAMP-A", "CAMP-B", "CAMP-B", "CAMP-A",
                    "CAMP-C", "CAMP-C", "CAMP-A", "CAMP-B", "CAMP-B"],
    "created_at": pd.date_range("2026-04-13 09:00", periods=9, freq="30min").tolist() + 
                  [pd.Timestamp("2028-01-01")],  # Issue 5: future timestamp (row 10)
})

print(f"Sample data: {len(calls)} records, 5 intentional quality issues")
calls

**You Should See:** 10 records. The 5 issues are:

| # | Issue | Row | Column |
|---|---|---|---|
| 1 | Null primary key | Row 7 (index 7) | call_id |
| 2 | Duplicate primary key | Rows 2 and 9 | call_id = C-003 |
| 3 | Invalid enum value | Row 5 | status = INVALID_STATUS |
| 4 | Negative value | Row 6 | duration = -50 |
| 5 | Future timestamp | Row 9 | created_at = 2028-01-01 |

---

## 3. Custom Python Quality Checks (No Framework)

Before using a framework, understand what quality checks actually do: run assertions against data and report pass/fail.

In [ ]:
from dataclasses import dataclass
from typing import Optional, List

@dataclass
class CheckResult:
    name: str
    passed: bool
    failing_rows: int = 0
    details: Optional[str] = None

def run_quality_checks(df: pd.DataFrame) -> List[CheckResult]:
    """Run quality checks. Returns list of pass/fail results."""
    results = []
    
    # Check 1: Primary key not null
    nulls = df["call_id"].isna().sum()
    results.append(CheckResult(
        name="call_id_not_null",
        passed=nulls == 0,
        failing_rows=nulls,
        details=f"{nulls} null call_ids" if nulls > 0 else None,
    ))
    
    # Check 2: Primary key unique
    dupes = df["call_id"].dropna().duplicated().sum()
    results.append(CheckResult(
        name="call_id_unique",
        passed=dupes == 0,
        failing_rows=dupes,
        details=f"{dupes} duplicate call_ids" if dupes > 0 else None,
    ))
    
    # Check 3: Valid status values
    valid = {"in-progress", "resolved", "missed", "voicemail", "transferred"}
    invalid = df[~df["status"].isin(valid)]
    results.append(CheckResult(
        name="status_valid",
        passed=len(invalid) == 0,
        failing_rows=len(invalid),
        details=f"Invalid: {invalid['status'].unique().tolist()}" if len(invalid) > 0 else None,
    ))
    
    # Check 4: Duration non-negative
    negative = df[df["duration"] < 0]
    results.append(CheckResult(
        name="duration_non_negative",
        passed=len(negative) == 0,
        failing_rows=len(negative),
        details=f"Min value: {df['duration'].min()}" if len(negative) > 0 else None,
    ))
    
    # Check 5: No future timestamps
    now = pd.Timestamp.now()
    future = df[df["created_at"] > now]
    results.append(CheckResult(
        name="no_future_timestamps",
        passed=len(future) == 0,
        failing_rows=len(future),
        details=f"Future dates: {future['created_at'].tolist()}" if len(future) > 0 else None,
    ))
    
    # Check 6: Row count reasonable
    count = len(df)
    results.append(CheckResult(
        name="row_count_minimum",
        passed=count >= 5,
        details=f"Only {count} rows" if count < 5 else None,
    ))
    
    return results

# Run checks
results = run_quality_checks(calls)

# Report
print("Data Quality Report")
print("=" * 60)
for r in results:
    icon = "PASS" if r.passed else "FAIL"
    print(f"  [{icon}] {r.name}", end="")
    if r.details:
        print(f" — {r.details}", end="")
    print()

passed = sum(1 for r in results if r.passed)
failed = sum(1 for r in results if not r.passed)
print(f"\nResult: {passed} passed, {failed} failed")

**You Should See:** 5 checks FAIL (null, duplicate, invalid status, negative duration, future timestamp) and 1 check PASS (row count). All 5 issues caught.

---

## 4. Quality Gate Pattern: Good Records vs DLQ

A quality gate splits records into good (continue to Silver) and bad (quarantine to DLQ).

In [ ]:
# WHY: Don't drop bad records. Don't crash on bad records.
# Route them to a quarantine table (Dead Letter Queue) for investigation.

VALID_STATUSES = {"in-progress", "resolved", "missed", "voicemail", "transferred"}

def quality_gate(df):
    """Split records into good (Silver) and bad (DLQ)."""
    errors = []
    for idx, row in df.iterrows():
        row_errors = []
        if pd.isna(row["call_id"]):
            row_errors.append("null_call_id")
        if row["status"] not in VALID_STATUSES:
            row_errors.append(f"invalid_status:{row['status']}")
        if row["duration"] < 0:
            row_errors.append(f"negative_duration:{row['duration']}")
        if pd.notna(row.get("created_at")) and row["created_at"] > pd.Timestamp.now():
            row_errors.append("future_timestamp")
        errors.append("; ".join(row_errors) if row_errors else None)
    
    df = df.copy()
    df["validation_error"] = errors
    
    good = df[df["validation_error"].isna()].drop(columns=["validation_error"])
    bad = df[df["validation_error"].notna()]
    
    # Deduplicate good records (keep first occurrence)
    good = good.drop_duplicates(subset=["call_id"], keep="first")
    
    return good, bad

good_records, dlq_records = quality_gate(calls)

print(f"Good records (→ Silver): {len(good_records)}")
print(f"Bad records (→ DLQ):     {len(dlq_records)}")
print()
print("DLQ contents:")
dlq_records[["call_id", "status", "duration", "validation_error"]]

**You Should See:** 6 good records go to Silver. 4 bad records go to DLQ (null call_id, invalid status, negative duration, future timestamp). The duplicate C-003 is handled by deduplication in the good path.

---

## 5. Great Expectations

Now the same checks using a production framework. Great Expectations gives you:
- 300+ built-in check types
- Auto-generated HTML reports (Data Docs)
- Integration with Airflow, Spark, and cloud warehouses

In [ ]:
import great_expectations as gx

# Create a data context (GX's configuration hub)
context = gx.get_context()

# Create a data source from our DataFrame
datasource = context.sources.add_or_update_pandas("calls_source")
data_asset = datasource.add_dataframe_asset("calls")
batch_request = data_asset.build_batch_request(dataframe=calls)

# Create a validator
validator = context.get_validator(
    batch_request=batch_request,
    create_expectation_suite_with_name="calls_quality_suite",
)

print("Great Expectations context ready")

In [ ]:
# Define expectations (same checks as our custom Python)

# Check 1: call_id not null
r1 = validator.expect_column_values_to_not_be_null("call_id")
print(f"call_id not null:     {'PASS' if r1.success else 'FAIL'} — {r1.result.get('unexpected_count', 0)} failures")

# Check 2: call_id unique
r2 = validator.expect_column_values_to_be_unique("call_id")
print(f"call_id unique:      {'PASS' if r2.success else 'FAIL'} — {r2.result.get('unexpected_count', 0)} failures")

# Check 3: status in valid set
r3 = validator.expect_column_values_to_be_in_set(
    "status", ["in-progress", "resolved", "missed", "voicemail", "transferred"]
)
print(f"status valid:        {'PASS' if r3.success else 'FAIL'} — {r3.result.get('unexpected_count', 0)} failures")

# Check 4: duration non-negative
r4 = validator.expect_column_values_to_be_between("duration", min_value=0, max_value=28800)
print(f"duration in range:   {'PASS' if r4.success else 'FAIL'} — {r4.result.get('unexpected_count', 0)} failures")

# Check 5: row count
r5 = validator.expect_table_row_count_to_be_between(min_value=5, max_value=100000)
print(f"row count:           {'PASS' if r5.success else 'FAIL'} — count={r5.result.get('observed_value', '?')}")

**You Should See:** Same results as our custom Python — 4 FAIL, 1 PASS. Great Expectations catches the same issues but with richer metadata (unexpected_count, unexpected_values, etc.).

---

## 6. Comparison: Custom vs Framework

Both approaches catch the same issues. The difference is in maintainability and reporting.

In [ ]:
comparison = """
+------------------------+-------------------------+------------------------------+
| Factor                 | Custom Python            | Great Expectations           |
+------------------------+-------------------------+------------------------------+
| Setup time             | Minutes                  | Hours (config, stores, etc.) |
| Check types            | You write each one       | 300+ built-in                |
| Reporting              | Print statements         | HTML Data Docs (auto)        |
| Airflow integration    | PythonOperator           | GXCheckpointOperator         |
| Profiling              | No                       | Yes (auto-discover checks)   |
| Team scalability       | Hard (tribal knowledge)  | Good (declarative configs)   |
| Dependencies           | None (just pandas)       | great_expectations package   |
+------------------------+-------------------------+------------------------------+
| RECOMMENDATION         | < 10 tables, small team  | > 10 tables, growing team    |
+------------------------+-------------------------+------------------------------+
"""

print(comparison)

---

## Summary

| What You Built | How It Works |
|---|---|
| Custom quality checks | Python functions, explicit pass/fail, full control |
| Quality gate (DLQ pattern) | Good records → Silver, bad records → quarantine |
| Great Expectations checks | Declarative expectations, richer reporting |

All three are cloud-agnostic. They run on pandas DataFrames, PySpark DataFrames, or SQL databases.

For cloud-native tools (Dataplex, Glue DQ, Synapse DQ), see the [Cloud Walkthroughs](../../playbooks/data/pipelines/data-quality/04_Cloud_Walkthroughs.md) chapter — those are console-configured, not notebook-runnable.

---

## Next Steps

- [Data Quality Tools - Tools Compared](../../playbooks/data/pipelines/data-quality/02_Tools_Compared.md) — Great Expectations vs dbt vs Soda vs cloud-native
- [Data Quality Tools - Cloud Walkthroughs](../../playbooks/data/pipelines/data-quality/04_Cloud_Walkthroughs.md) — Dataplex, Glue DQ, Synapse DQ setup
- [ETL/ELT Patterns - Quality](../../playbooks/data/pipelines/etl-elt/08_Quality_Security_Governance.md) — PII masking, schema drift, audit trail

---

[Community](https://www.skool.com/deliverymomentum) | [Book a 1:1](https://calendly.com/sunil-mogadati/connect)